In [1]:
# 1. Install required libraries
!pip install pandas numpy joblib scikit-learn==1.8.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 39.2 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [2]:
# 2. Import libraries
import pandas as pd
import joblib

from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error

In [3]:
# 3. Mount Google Drive
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# 4. Load dataset from Google Drive

DATASET_PATH = "/content/drive/MyDrive/archive/Body Measurements _ original_CSV.csv"

df = pd.read_csv(DATASET_PATH)

# Clean column names
df.columns = df.columns.str.strip()

df.head()

,Gender,Age,HeadCircumference,ShoulderWidth,ChestWidth,Belly,Waist,Hips,ArmLength,ShoulderToWaist,WaistToKnee,LegLength,TotalHeight
0,1.0,30,22,18,20,18,14,22,22,25,25,22,52
1,1.0,28,19,22,17,18,21,25,28,23,25,20,56
2,2.0,27,21,18,16,14,10,15,21,18,14,18,53
3,1.0,29,20,20,18,11,19,14,24,21,20,21,45
4,2.0,28,16,14,18,13,11,30,25,22,32,13,47


In [5]:
# 5. Rename columns
df = df.rename(columns={
    "Gender": "gender",
    "TotalHeight": "height",
    "ShoulderWidth": "shoulder_width",
    "Waist": "waist",
    "LegLength": "leg_length",
})

df.head()

,gender,Age,HeadCircumference,shoulder_width,ChestWidth,Belly,waist,Hips,ArmLength,ShoulderToWaist,WaistToKnee,leg_length,height
0,1.0,30,22,18,20,18,14,22,22,25,25,22,52
1,1.0,28,19,22,17,18,21,25,28,23,25,20,56
2,2.0,27,21,18,16,14,10,15,21,18,14,18,53
3,1.0,29,20,20,18,11,19,14,24,21,20,21,45
4,2.0,28,16,14,18,13,11,30,25,22,32,13,47


In [6]:
# 6. Select required columns
required_columns = [
    "gender",
    "height",
    "shoulder_width",
    "waist",
    "leg_length",
]

df = df[required_columns]
df = df.dropna()

df.head()

,gender,height,shoulder_width,waist,leg_length
0,1.0,52,18,14,22
1,1.0,56,22,21,20
2,2.0,53,18,10,18
3,1.0,45,20,19,21
4,2.0,47,14,11,13


In [7]:
# 7. Define input and output

# Input: height + gender
X = df[["height", "gender"]]

# Output: predicted body measurements
y = df[[
    "shoulder_width",
    "waist",
    "leg_length",
]]

In [8]:
# 8. Preprocess gender and train model

preprocessor = ColumnTransformer(
    transformers=[
        ("gender", OneHotEncoder(handle_unknown="ignore"), ["gender"]),
    ],
    remainder="passthrough"
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", RandomForestRegressor(
            n_estimators=200,
            random_state=42
        ))
    ]
)

In [9]:
# 9. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('gender', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [10]:
# 10. Evaluate model
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)

print("Mean Absolute Error:", mae)

Mean Absolute Error: 4.199273044824364


In [11]:
# 11. Test prediction
sample = pd.DataFrame([{
    "height": 170,
    "gender": "male"
}])

sample_prediction = model.predict(sample)[0]

print("Predicted Shoulder Width:", round(sample_prediction[0], 2))
print("Predicted Waist:", round(sample_prediction[1], 2))
print("Predicted Leg Length:", round(sample_prediction[2], 2))

Predicted Shoulder Width: 19.81
Predicted Waist: 31.36
Predicted Leg Length: 41.7


In [12]:
# 12. Save model
joblib.dump(model, "body_measurement_model.pkl")

print("Model saved successfully")

Model saved successfully


In [13]:
# 13. Download model
files.download("body_measurement_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>